In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [10]:
conn = sqlite3.connect('../db.sqlite3')

query = """
SELECT
    oi.id AS order_item_id,
    oi.quantity,
    o.id AS order_id,
    o.datetime,
    p.id AS product_id,
    p.name AS product_name,
    p.price
FROM restaurant_orderitem oi
INNER JOIN restaurant_order o ON oi.order_id = o.id
INNER JOIN restaurant_product p ON oi.product_id = p.id
"""

df_joined = pd.read_sql_query(query, conn)
conn.close()

df_joined.to_csv('combined_data.csv', index=False)

df = pd.read_csv('combined_data.csv')

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [12]:
df = pd.read_csv('combined_data.csv')
top_10 = (
    df.groupby('product_name')['quantity']
    .sum()
    .reset_index()
    .sort_values(by='quantity', ascending=False)
    .head(10)
)
def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f'{pct:.1f}% ({val})'
    return my_autopct

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 8), facecolor='#1e1e1e')
ax.set_facecolor('#1e1e1e')

colors = ['#3a86c8', '#aa6c39', '#2d9a47', '#a62b2b', '#7b4fa3',
          '#8a6d3b', '#4a7ebb', '#be527f', '#777777', '#5f9ea0']

# 4. Построение диаграммы
wedges, texts, autotexts = ax.pie(
    top_10['quantity'],
    labels=top_10['product_name'],
    autopct=make_autopct(top_10['quantity']),
    startangle=140,
    colors=colors,
    textprops=dict(color="w", fontsize=9)
)

for autotext in autotexts:
    autotext.set_color('#aaaaaa')

ax.set_title('Top 10 positions in menu by quantity', color='w', fontsize=12, pad=20)
ax.set_ylabel('Quantity', color='w', fontsize=10)

plt.savefig('../demo/pie.png', bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [13]:
df['item_price'] = df['price'] * df['quantity']

top_10_revenue = (
    df.groupby('product_name')['item_price']
    .sum()
    .reset_index()
    .sort_values(by='item_price', ascending=False)
    .head(10)
)

def make_revenue_pct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f'{pct:.1f}% ({val})'
    return my_autopct

plt.figure(figsize=(10, 8))
plt.pie(
    top_10_revenue['item_price'],
    labels=top_10_revenue['product_name'],
    autopct=make_revenue_pct(top_10_revenue['item_price']),
    startangle=140
)

plt.title('Top 10 positions in menu by revenue')
plt.ylabel('Item Price')
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [14]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['order_hour'] = df['datetime'].dt.hour

hourly_revenue = df.groupby('order_hour')['item_price'].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(
    hourly_revenue['order_hour'],
    hourly_revenue['item_price'],
    color='#3a86c8',
    width=0.6
)

plt.title('Profit by Order Hour')
plt.xlabel('Order Hour')
plt.ylabel('Total Income')

plt.xticks(range(24))
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [18]:
df['day_of_week'] = df['datetime'].dt.dayofweek

daily_revenue = df.groupby('day_of_week')['item_price'].sum().reset_index()

days_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 6))
plt.bar(
    daily_revenue['day_of_week'],
    daily_revenue['item_price'],
    color='#3a86c8',
    width=0.6
)

plt.title('Profit by Day of the Week')
plt.xlabel('Order Day Of The Week')
plt.ylabel('Total Income')

plt.xticks(ticks=range(7), labels=days_labels)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()